In [1]:
import time
import numpy as np
import sys
import pandas as pd
import nrm
import csv
import subprocess
import os
import tarfile
import random
from datetime import datetime
import torch

In [2]:
def normalize(data, MIN, MAX):
    return np.round((np.float64(data) - MIN) / (MAX - MIN), decimals=4)

class FCNetwork(torch.nn.Module):
  def __init__(self, layers=[20,20]):
    super(FCNetwork, self).__init__()
    # self.all_observations = torch.tensor(stack_observations(env), dtype=torch.float32)
    dim_input = 5
    dim_output = 16
    net_layers = []

    dim = dim_input
    for i, layer_size in enumerate(layers):
      net_layers.append(torch.nn.Linear(dim, layer_size))
      net_layers.append(torch.nn.ReLU())
      dim = layer_size
    net_layers.append(torch.nn.Linear(dim, dim_output))
    self.layers = net_layers
    self.network = torch.nn.Sequential(*net_layers)

  def forward(self, states):
    # observations = torch.index_select(self.all_observations, 0, states)
    states_tensor = torch.tensor(states, dtype=torch.float32)  # Ensure the correct dtype
    return self.network(states_tensor)

  def print_weights(self):
    for name, param in self.named_parameters():
        if param.requires_grad:
            print(f"{name}: {param.data.numpy()}")
            
model = FCNetwork(layers=[5,5])

i = 0
# APPLICATIONS = ['ones-npb-ep', 'ones-npb-is','ones-stream-full', 'ones-stream-triad', 'ones-stream-add', 'ones-stream-copy', 'ones-stream-scale', 'phases-stream-full']
# APPLICATIONS = ['ones-npb-is']
APPLICATIONS = ['ones-npb-ep', 'ones-stream-full']
policy_folder = '/home/cc/summer2024/main_codes/'  # Default policy file
# policy_file = os.path.join(policy_folder,'BCQ_SYS_0_20240929_183736.pt')
while i < len(sys.argv):
    if sys.argv[i] == '--application':
        APPLICATION = sys.argv[i+1]
        i += 1
    elif sys.argv[i] == '--policy':
        policy_name = sys.argv[i+1]  # Update policy file from argument
        policy_file = os.path.join(policy_folder, policy_name)
        i += 1
    i +=1

In [3]:
def get_data_dir(subfolder):
    current_dir = os.getcwd()
    print(current_dir)
    return os.path.join(current_dir, "experiment_data", f"{subfolder}")

DATA_DIR = get_data_dir("training_data")

csv_file_path = f'{DATA_DIR}/training_dataset.csv'


/home/cc/summer2024/main_codes


In [4]:
model = FCNetwork(layers=[10,10])
policy_name = "trained_network_weights_20250620_045521_0.3_0.0001.pth"
policy_file = os.path.join("/home/cc/summer2024/main_codes/trained_models", policy_name)
model.load_state_dict(torch.load(policy_file))
model.eval()

FCNetwork(
  (network): Sequential(
    (0): Linear(in_features=5, out_features=10, bias=True)
    (1): ReLU()
    (2): Linear(in_features=10, out_features=10, bias=True)
    (3): ReLU()
    (4): Linear(in_features=10, out_features=16, bias=True)
  )
)

In [5]:
data = pd.read_csv(csv_file_path)

df = pd.DataFrame(data)
ACTIONS = [78.0, 83.0, 89.0, 95.0, 101.0, 107.0, 112.0, 118.0, 124.0, 130.0, 136.0, 141.0, 147.0, 153.0, 159.0, 165.0]


for i in range(len(df)):
    # print(df.iloc[i])
    state = np.array(df.iloc[i][1:6],dtype=np.float32)
    suggested_action = model(state)
    argmax = (np.argmax(suggested_action.detach().numpy(),axis=-1))
    # print(suggested_action,"\n",argmax,ACTIONS[argmax])
    print(f"{i+1},.....,{df.iloc[i][0]}.....{ACTIONS[argmax]}")




1,.....,ones-stream-triad.....112.0
2,.....,ones-stream-triad.....95.0
3,.....,ones-stream-triad.....107.0
4,.....,ones-stream-triad.....95.0
5,.....,ones-stream-triad.....107.0
6,.....,ones-stream-triad.....95.0
7,.....,ones-stream-triad.....95.0
8,.....,ones-stream-triad.....95.0
9,.....,ones-stream-triad.....107.0
10,.....,ones-stream-triad.....95.0
11,.....,ones-stream-triad.....95.0
12,.....,ones-stream-triad.....95.0
13,.....,ones-stream-triad.....107.0
14,.....,ones-stream-triad.....95.0
15,.....,ones-stream-triad.....95.0
16,.....,ones-stream-triad.....95.0
17,.....,ones-stream-triad.....95.0
18,.....,ones-stream-triad.....95.0
19,.....,ones-stream-triad.....95.0
20,.....,ones-stream-triad.....95.0
21,.....,ones-stream-triad.....95.0
22,.....,ones-stream-triad.....107.0
23,.....,ones-stream-triad.....107.0
24,.....,ones-stream-triad.....95.0
25,.....,ones-stream-triad.....107.0
26,.....,ones-stream-triad.....95.0
27,.....,ones-stream-triad.....107.0
28,.....,ones-stream-triad..

/tmp/ipykernel_179717/1472399183.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(f"{i+1},.....,{df.iloc[i][0]}.....{ACTIONS[argmax]}")


421,.....,ones-npb-ep.....153.0
422,.....,ones-npb-ep.....153.0
423,.....,ones-npb-ep.....141.0
424,.....,ones-npb-ep.....141.0
425,.....,ones-npb-ep.....153.0
426,.....,ones-npb-ep.....153.0
427,.....,ones-npb-ep.....153.0
428,.....,ones-npb-ep.....153.0
429,.....,ones-npb-ep.....153.0
430,.....,ones-npb-ep.....141.0
431,.....,ones-npb-ep.....153.0
432,.....,ones-npb-ep.....153.0
433,.....,ones-npb-ep.....153.0
434,.....,ones-npb-ep.....141.0
435,.....,ones-npb-ep.....141.0
436,.....,ones-npb-ep.....141.0
437,.....,ones-npb-ep.....141.0
438,.....,ones-npb-ep.....153.0
439,.....,ones-npb-ep.....153.0
440,.....,ones-npb-ep.....153.0
441,.....,ones-npb-ep.....153.0
442,.....,ones-npb-ep.....153.0
443,.....,ones-npb-ep.....153.0
444,.....,ones-npb-ep.....141.0
445,.....,ones-npb-ep.....153.0
446,.....,ones-npb-ep.....153.0
447,.....,ones-npb-ep.....141.0
448,.....,ones-npb-ep.....153.0
449,.....,ones-npb-ep.....153.0
450,.....,ones-npb-ep.....153.0
451,.....,ones-npb-ep.....153.0
452,....